# Hospital Readmission Analysis: Diabetic Patients (130 US Hospitals)

## Problem statement
When a diabetic patient returns to hospital within 30 days of being discharged, it is bad for the patient's health and expensive for the hospital. This project analyses about 100,000 hospital visits from 130 US hospitals (1999–2008) to find which patient details, hospital-stay patterns and treatments are linked to **early (30-day) readmission**, so hospitals know which patients need extra follow-up care.

## Questions this analysis answers
1. What percentage of visits end in a 30-day readmission, and how does this change with **age**?
2. Is the **length of the hospital stay** related to early readmission?
3. Are patients with more **previous inpatient visits** more likely to come back?
4. Does **where the patient goes after discharge** (home, care facility, etc.) matter?
5. How do **diabetes treatment factors** (A1C testing, insulin changes, number of medications) relate to readmission?
6. Which **primary diagnoses** have the highest readmission rates?

**Data source:** "Diabetes 130-US hospitals for years 1999–2008", UCI Machine Learning Repository / Kaggle. Loaded into the MySQL database `hospital_readmission` by `load_data.py`.

**Workflow:** MySQL → pandas (cleaning + analysis) → Plotly & Cufflinks (visuals) → observations → Streamlit dashboard.

## 1. Setup
Importing the libraries.

> **Compatibility note:** Cufflinks is an older library. With NumPy 2.x it builds colour strings like `rgba(255, 153, 51, np.float64(1.0))`, which Plotly rejects. The line `np.set_printoptions(legacy="1.25")` fixes this, and `requirements.txt` pins Plotly to 5.24.1 because Plotly 6 removed a property Cufflinks still uses.

In [12]:
import sys
from pathlib import Path

import numpy as np
np.set_printoptions(legacy="1.25")       # needed for Cufflinks with NumPy 2.x

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import cufflinks as cf

# Make config.py (one folder up) importable
sys.path.insert(0, str(Path.cwd().parent))
from config import get_engine

cf.go_offline()                          # Cufflinks: draw charts offline
cf.set_config_file(theme="white")
pd.set_option("display.max_columns", 60)

# Colours used throughout the notebook
BLUE, RED, GREY = "#1F5A85", "#C0392B", "#8A99A8"
print("Libraries loaded.")

Libraries loaded.


## 2. Load the data from MySQL
Instead of reading the CSV, we read from MySQL. The query **JOINs** the main table with the three lookup tables so ID numbers become readable text (for example `admission_type_id = 1` becomes *Emergency*).

In [13]:
engine = get_engine()

query = """
SELECT  e.*,
        t.description AS admission_type,
        d.description AS discharge_disposition,
        s.description AS admission_source
FROM    patient_encounters e
LEFT JOIN admission_types        t ON e.admission_type_id        = t.admission_type_id
LEFT JOIN discharge_dispositions d ON e.discharge_disposition_id = d.discharge_disposition_id
LEFT JOIN admission_sources      s ON e.admission_source_id      = s.admission_source_id
"""

raw = pd.read_sql(query, engine)
print(f"Loaded {raw.shape[0]:,} rows and {raw.shape[1]} columns from MySQL")

Loaded 101,766 rows and 53 columns from MySQL


## 3. First look at the data

In [14]:
raw.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,a1c_result,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide_metformin,glipizide_metformin,glimepiride_pioglitazone,metformin_rosiglitazone,metformin_pioglitazone,med_change,diabetes_med,readmitted,admission_type,discharge_disposition,admission_source
0,12522,48330783,Caucasian,Female,[80-90),NaN,2,1,4,13,NaN,NaN,68,2,28,0,0,0,398,427,38,8,None,None,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO,Urgent,Discharged to home,Transfer from a hospital
1,15738,63555939,Caucasian,Female,[90-100),NaN,3,3,4,12,NaN,InternalMedicine,33,3,18,0,0,0,434,198,486,8,None,None,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO,Elective,Discharged/transferred to SNF,Transfer from a hospital
2,16680,42519267,Caucasian,Male,[40-50),NaN,1,1,7,1,NaN,NaN,51,0,8,0,0,0,197,157,250,5,None,None,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO,Emergency,Discharged to home,Emergency Room
3,28236,89869032,AfricanAmerican,Female,[40-50),NaN,1,1,7,9,NaN,NaN,47,2,17,0,0,0,250.7,403,996,9,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,Yes,>30,Emergency,Discharged to home,Emergency Room
4,35754,82637451,Caucasian,Male,[50-60),NaN,2,1,2,3,NaN,NaN,31,6,16,0,0,0,414,411,250,9,None,None,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,No,Yes,>30,Urgent,Discharged to home,Clinic Referral


In [15]:
raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 53 columns):
 #   Column                    Non-Null Count   Dtype
---  ------                    --------------   -----
 0   encounter_id              101766 non-null  int64
 1   patient_nbr               101766 non-null  int64
 2   race                      99493 non-null   str  
 3   gender                    101766 non-null  str  
 4   age                       101766 non-null  str  
 5   weight                    3197 non-null    str  
 6   admission_type_id         101766 non-null  int64
 7   discharge_disposition_id  101766 non-null  int64
 8   admission_source_id       101766 non-null  int64
 9   time_in_hospital          101766 non-null  int64
 10  payer_code                61510 non-null   str  
 11  medical_specialty         51817 non-null   str  
 12  num_lab_procedures        101766 non-null  int64
 13  num_procedures            101766 non-null  int64
 14  num_medications           10176

In [16]:
raw[["time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications",
     "number_outpatient", "number_emergency", "number_inpatient", "number_diagnoses"]].describe().round(2)

,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses
count,101766.00,101766.00,101766.00,101766.00,101766.00,101766.00,101766.00,101766.00
mean,4.40,43.10,1.34,16.02,0.37,0.20,0.64,7.42
std,2.99,19.67,1.71,8.13,1.27,0.93,1.26,1.93
min,1.00,1.00,0.00,1.00,0.00,0.00,0.00,1.00
25%,2.00,31.00,0.00,10.00,0.00,0.00,0.00,6.00
50%,4.00,44.00,1.00,15.00,0.00,0.00,0.00,8.00
75%,6.00,57.00,2.00,20.00,0.00,0.00,1.00,9.00
max,14.00,132.00,6.00,81.00,42.00,76.00,21.00,16.00


**What we see:** Numeric columns look sensible (stay of 1–14 days, 1–81 medications). Some text columns have missing values, and a few columns use codes that need to be translated or grouped before analysis.

## 4. Data cleaning
We work on a copy called `df` so the raw data stays untouched.

### 4.1 Missing values

In [17]:
df = raw.copy()

missing = (df.isna().mean() * 100).round(2)
missing = missing[missing > 0].sort_values(ascending=False).to_frame("missing_pct")
missing

,missing_pct
weight,96.86
medical_specialty,49.08
payer_code,39.56
race,2.23
diag_3,1.40
diag_2,0.35
diag_1,0.02


In [18]:
fig = missing.iplot(kind="bar", asFigure=True, colors=[BLUE],
                    title="Percentage of missing values per column",
                    xTitle="Column", yTitle="% missing")
fig.show()

**Observation:** `weight` is about 97% empty and `payer_code` (insurance type) about 40% empty, so they cannot be used reliably. `medical_specialty` is 49% empty, but it is still useful for the rest, so we keep it and label blanks as *Unknown*. `race` is only 2% missing.

**Decisions:**
- Drop `weight` and `payer_code` (too empty, not needed for our questions)
- Fill missing `medical_specialty` and `race` with *Unknown*
- Leave `diag_2` / `diag_3` as they are; we only use `diag_1` (the primary diagnosis)

In [19]:
df = df.drop(columns=["weight", "payer_code"])
df["medical_specialty"] = df["medical_specialty"].fillna("Unknown")
df["race"] = df["race"].fillna("Unknown")
print("Remaining missing values:\n", df.isna().sum()[df.isna().sum() > 0])

Remaining missing values:
 diag_1      21
diag_2     358
diag_3    1423
dtype: int64


### 4.2 Invalid values and rows that don't belong
- 3 rows have gender `Unknown/Invalid`, which we remove.
- Patients who **died or went to hospice** cannot be readmitted. Keeping them would make the readmission rate look lower than it really is, so we remove them (discharge IDs 11, 13, 14, 19, 20, 21).

In [20]:
before = len(df)
df = df[df["gender"] != "Unknown/Invalid"]
print("Invalid gender rows removed:", before - len(df))

before = len(df)
expired_or_hospice = [11, 13, 14, 19, 20, 21]
df = df[~df["discharge_disposition_id"].isin(expired_or_hospice)]
print("Expired / hospice rows removed:", before - len(df))
print("Rows remaining:", f"{len(df):,}")

Invalid gender rows removed: 3
Expired / hospice rows removed: 2423
Rows remaining: 99,340


### 4.3 Duplicates

In [21]:
print("Duplicate encounter IDs:", df["encounter_id"].duplicated().sum())
print("Fully duplicated rows  :", df.drop(columns="encounter_id").duplicated().sum())
print("Unique patients        :", f"{df['patient_nbr'].nunique():,}")

Duplicate encounter IDs: 0
Fully duplicated rows  : 0
Unique patients        : 69,987


**Observation:** There are no duplicate visits. However, about 70,000 unique patients account for 99,000 visits, so some patients came many times. That is real behaviour (not an error), and it turns out to be one of the most important signals in the data, so we keep these rows.

### 4.4 Fixing data types and creating new columns
| New column | Why |
|---|---|
| `age_group` | `[70-80)` becomes `70-80`, which is easier to read in charts |
| `readmitted_30` | 1 if readmitted within 30 days, else 0. The **average** of this column is the readmission rate |
| `readmit_status` | Readable label for the three outcomes |
| `primary_diagnosis` | Groups 700+ ICD-9 codes into 9 disease groups |
| `a1c_result`, `max_glu_serum` | The text `None` actually means *test not done*, so we rename it |
| `a1c_tested` | Simple Tested / Not tested flag |
| `total_prior_visits` | Outpatient + emergency + inpatient visits in the previous year |
| `discharge_group`, `admission_group` | Fewer, clearer categories than the 30 original codes |

In [22]:
df["age_group"] = df["age"].str.strip("[)")
df["readmitted_30"] = (df["readmitted"] == "<30").astype(int)
df["readmit_status"] = df["readmitted"].map({
    "<30": "Within 30 days", ">30": "After 30 days", "NO": "Not readmitted"})

df["a1c_result"] = df["a1c_result"].replace({"None": "Not tested"})
df["max_glu_serum"] = df["max_glu_serum"].replace({"None": "Not tested"})
df["a1c_tested"] = np.where(df["a1c_result"] == "Not tested", "Not tested", "Tested")

df["total_prior_visits"] = df["number_outpatient"] + df["number_emergency"] + df["number_inpatient"]

In [23]:
def icd9_group(code):
    """Convert an ICD-9 diagnosis code into a disease group."""
    if pd.isna(code):
        return "Missing"
    if code.startswith(("V", "E")):          # supplementary codes
        return "Other"
    value = float(code)
    if 390 <= value <= 459 or int(value) == 785: return "Circulatory"
    if 460 <= value <= 519 or int(value) == 786: return "Respiratory"
    if 520 <= value <= 579 or int(value) == 787: return "Digestive"
    if int(value) == 250:                        return "Diabetes"
    if 800 <= value <= 999:                      return "Injury"
    if 710 <= value <= 739:                      return "Musculoskeletal"
    if 580 <= value <= 629 or int(value) == 788: return "Genitourinary"
    if 140 <= value <= 239:                      return "Neoplasms"
    return "Other"


def discharge_group(i):
    if i == 1:                return "Home"
    if i in (6, 8):           return "Home with health service"
    if i in (3, 4, 5, 15, 22, 23, 24, 27, 28, 29, 30):
                              return "Care / rehab facility"
    if i in (2, 10):          return "Another hospital"
    if i == 7:                return "Left against advice"
    return "Other / Unknown"


df["primary_diagnosis"] = df["diag_1"].apply(icd9_group)
df["discharge_group"] = df["discharge_disposition_id"].apply(discharge_group)
df["admission_group"] = (df["admission_type_id"]
                         .map({1: "Emergency", 2: "Urgent", 3: "Elective", 7: "Trauma Center"})
                         .fillna("Other / Unknown"))

df[["age", "age_group", "readmitted", "readmitted_30", "diag_1",
    "primary_diagnosis", "discharge_group", "a1c_result"]].head()

,age,age_group,readmitted,readmitted_30,diag_1,primary_diagnosis,discharge_group,a1c_result
0,[80-90),80-90,NO,0,398,Circulatory,Home,Not tested
1,[90-100),90-100,NO,0,434,Circulatory,Care / rehab facility,Not tested
2,[40-50),40-50,NO,0,197,Neoplasms,Home,Not tested
3,[40-50),40-50,>30,0,250.7,Diabetes,Home,Not tested
4,[50-60),50-60,>30,0,414,Circulatory,Home,Not tested


### 4.5 Outliers

In [24]:
num_cols = ["time_in_hospital", "num_lab_procedures", "num_medications"]
# Box plots store every point inside the notebook, so we draw them from a
# random sample of 10,000 rows. The shape is the same; the file stays small.
sample = df.sample(10_000, random_state=42)
long_df = sample[num_cols].melt(var_name="column", value_name="value")
fig = px.box(long_df, x="column", y="value", color="column",
             title="Spread of key numeric columns (10,000-row sample, dots = outliers)",
             color_discrete_sequence=[BLUE, GREY, RED])
fig.update_layout(showlegend=False)
fig.show()

for col in num_cols:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    n_out = ((df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)).sum()
    print(f"{col:<20} outliers (IQR rule): {n_out:,}")

time_in_hospital     outliers (IQR rule): 2,147
num_lab_procedures   outliers (IQR rule): 120
num_medications      outliers (IQR rule): 2,455


**Observation:** Some patients have very high values (e.g. up to 81 medications or 132 lab tests). These are **medically possible** for very sick patients, not typing mistakes, so we **keep** them. Removing them would hide exactly the complex patients hospitals care most about.

In [25]:
print("Final clean dataset:", df.shape)

Final clean dataset: (99340, 59)


## 5. Exploratory analysis and visuals
To compare groups we use the **30-day readmission rate**: the percentage of visits in a group that ended in a readmission within 30 days. This helper function calculates it for any column.

In [26]:
def readmit_rate(data, column):
    """Visits and 30-day readmission rate (%) for each value of a column."""
    return (data.groupby(column, observed=True)
                .agg(visits=("readmitted_30", "size"),
                     readmit_rate=("readmitted_30", "mean"))
                .assign(readmit_rate=lambda t: (t["readmit_rate"] * 100).round(2)))

overall_rate = df["readmitted_30"].mean() * 100
print(f"Overall 30-day readmission rate: {overall_rate:.2f}%")

Overall 30-day readmission rate: 11.39%


### Q1a. How many patients come back?

In [27]:
status = df["readmit_status"].value_counts().reset_index()
status.columns = ["status", "visits"]

fig = status.iplot(kind="pie", labels="status", values="visits", asFigure=True,
                   colors=[GREY, BLUE, RED], hole=0.4,
                   title="Outcome of each hospital visit")
fig.show()

**Observation:** About **11.4%** of visits (roughly 1 in 9) end in a readmission within 30 days, and another 36% are readmitted later. More than half are not readmitted at all. The 30-day group is the smallest, which makes it harder to predict but very costly.

### Q1b. Readmission rate by age group

In [28]:
age_rate = readmit_rate(df, "age_group")

fig = age_rate["readmit_rate"].iplot(kind="bar", asFigure=True, colors=[BLUE],
                                     title="30-day readmission rate by age group",
                                     xTitle="Age group", yTitle="Readmission rate (%)")
fig.add_hline(y=overall_rate, line_dash="dash", line_color=RED,
              annotation_text=f"Average {overall_rate:.1f}%")
fig.show()
age_rate

,visits,readmit_rate
age_group,,
0-10,160,1.88
10-20,690,5.80
20-30,1649,14.31
30-40,3764,11.26
40-50,9607,10.66
50-60,17060,9.77
60-70,22058,11.30
70-80,25329,12.06
80-90,16434,12.57


**Observation:** From age 50 upward the rate rises steadily, from 9.8% (50–60) to 12.6% (80–90). Patients aged 60–90 are also the **largest groups** (about 64,000 visits), so they matter most. The 20–30 group shows 14.3%, but it has only about 1,600 visits, so that number is less reliable.

### Q2. Does a longer hospital stay mean more readmissions?

In [29]:
fig = px.histogram(df, x="time_in_hospital", color="readmit_status", barmode="group",
                   title="Length of stay by outcome",
                   labels={"time_in_hospital": "Days in hospital", "readmit_status": "Outcome"},
                   color_discrete_map={"Within 30 days": RED, "After 30 days": BLUE,
                                       "Not readmitted": GREY})
fig.show()

In [30]:
stay_rate = readmit_rate(df, "time_in_hospital")

fig = stay_rate["readmit_rate"].iplot(kind="line", asFigure=True, colors=[RED], width=3,
                                      title="30-day readmission rate by length of stay",
                                      xTitle="Days in hospital", yTitle="Readmission rate (%)")
fig.show()

**Observation:** Most stays are short (1–4 days). The readmission rate climbs from **8.4% for a 1-day stay** to around **14–15% for 8–10 days**, then flattens. Longer stays usually mean sicker patients, who are more likely to return.

### Q3. Do previous hospital stays predict readmission?

In [31]:
df["prior_inpatient_band"] = df["number_inpatient"].clip(upper=5).astype(str).replace({"5": "5+"})
inp_rate = readmit_rate(df, "prior_inpatient_band")

fig = inp_rate["readmit_rate"].iplot(kind="bar", asFigure=True, colors=[RED],
                                     title="30-day readmission rate by inpatient visits in the previous year",
                                     xTitle="Inpatient visits in previous year", yTitle="Readmission rate (%)")
fig.show()
inp_rate

,visits,readmit_rate
prior_inpatient_band,,
0,66242,8.59
1,18984,13.25
2,7300,17.93
3,3271,21.03
4,1574,24.14
5+,1969,37.13


**Observation:** This is the **strongest pattern in the whole dataset**. Patients with no inpatient stays in the previous year are readmitted 8.6% of the time; with 5 or more stays the rate is **37.1%**, more than four times higher. Past hospital use is the clearest warning sign.

### Q4. Does the discharge destination matter?

In [32]:
disc_rate = readmit_rate(df, "discharge_group").sort_values("readmit_rate")

fig = px.bar(disc_rate.reset_index(), x="readmit_rate", y="discharge_group", orientation="h",
             text="readmit_rate", color="readmit_rate", color_continuous_scale="Reds",
             title="30-day readmission rate by discharge destination",
             labels={"readmit_rate": "Readmission rate (%)", "discharge_group": ""})
fig.update_traces(texttemplate="%{text:.1f}%")
fig.show()
disc_rate

,visits,readmit_rate
discharge_group,,
Home,60232,9.30
Other / Unknown,4729,11.88
Home with health service,13010,12.71
Left against advice,623,14.45
Another hospital,2134,16.03
Care / rehab facility,18612,16.47


**Observation:** Patients sent **home** have the lowest rate (9.3%). Patients moved to a **care or rehab facility** (16.5%) or **another hospital** (16.0%) are much more likely to return, as they are usually frailer. Patients who left **against medical advice** are also high (14.5%).

### Q5a. Insulin changes and A1C testing

In [33]:
insulin_rate = readmit_rate(df, "insulin").reindex(["No", "Steady", "Up", "Down"])
fig = insulin_rate["readmit_rate"].iplot(kind="bar", asFigure=True, colors=[BLUE],
                                         title="30-day readmission rate by insulin status",
                                         xTitle="Insulin during the visit", yTitle="Readmission rate (%)")
fig.show()
insulin_rate

,visits,readmit_rate
insulin,,
No,46376,10.21
Steady,30069,11.37
Up,10987,13.33
Down,11908,14.22


**Observation:** When the insulin dose was **changed** during the visit, readmission is higher: 13.3% when increased and 14.2% when decreased, compared to 10.2% for patients not on insulin. A dose change suggests blood sugar was hard to control.

In [34]:
a1c = readmit_rate(df, "a1c_result").reindex(["Not tested", "Norm", ">7", ">8"])
a1c["share_of_visits_pct"] = (a1c["visits"] / a1c["visits"].sum() * 100).round(1)

fig = a1c[["readmit_rate"]].iplot(kind="bar", asFigure=True, colors=[GREY],
                                  title="30-day readmission rate by A1C test result",
                                  xTitle="A1C result", yTitle="Readmission rate (%)")
fig.show()
a1c

,visits,readmit_rate,share_of_visits_pct
a1c_result,,,
Not tested,82506,11.69,83.1
Norm,4922,9.77,5.0
>7,3775,10.15,3.8
>8,8137,9.94,8.2


**Observation:** The A1C blood-sugar test was **not done in 83% of visits**. Patients who were tested had a lower readmission rate (about 9.8–10.2%) than those who were not (11.7%), whatever their result. Testing may lead to better diabetes management before discharge.

### Q5b. Number of medications

In [35]:
sample = df.sample(10_000, random_state=42)       # sample keeps the notebook small
fig = px.box(sample, x="readmit_status", y="num_medications", color="readmit_status",
             category_orders={"readmit_status": ["Not readmitted", "After 30 days", "Within 30 days"]},
             color_discrete_map={"Within 30 days": RED, "After 30 days": BLUE, "Not readmitted": GREY},
             title="Number of medications by outcome (10,000-row sample)",
             labels={"readmit_status": "Outcome", "num_medications": "Number of medications"})
fig.update_layout(showlegend=False)
fig.show()

In [36]:
med_rate = readmit_rate(df, "num_medications").reset_index()
med_rate = med_rate[med_rate["visits"] >= 100]      # ignore very small groups

fig = px.scatter(med_rate, x="num_medications", y="readmit_rate", size="visits",
                 color_discrete_sequence=[BLUE],
                 title="Readmission rate vs number of medications (bubble size = visits)",
                 labels={"num_medications": "Number of medications", "readmit_rate": "Readmission rate (%)"})
fig.show()

**Observation:** Patients readmitted within 30 days take slightly more medications (median 16 vs 14 for those not readmitted). The scatter shows the rate rising from about 4–9% for patients on fewer than 8 medications to about 12–14% for 17–30 medications, after which the small groups make it noisy. More medications usually means more health problems.

### Q6. Which primary diagnoses are most risky?

In [37]:
diag_rate = readmit_rate(df[df["primary_diagnosis"] != "Missing"], "primary_diagnosis")
diag_rate = diag_rate.sort_values("readmit_rate", ascending=True)

fig = diag_rate["readmit_rate"].iplot(kind="barh", asFigure=True, colors=[BLUE],
                                      title="30-day readmission rate by primary diagnosis",
                                      xTitle="Readmission rate (%)")
fig.show()
diag_rate.sort_values("visits", ascending=False)

,visits,readmit_rate
primary_diagnosis,,
Circulatory,29680,11.69
Other,17793,11.68
Respiratory,13934,10.06
Digestive,9333,10.82
Diabetes,8661,13.10
Injury,6851,12.41
Genitourinary,5002,11.04
Musculoskeletal,4935,9.54
Neoplasms,3131,10.89


**Observation:** Patients admitted mainly **for diabetes itself** have the highest rate (13.1%), followed by **injuries** (12.4%). **Circulatory** problems (heart and blood vessels) are by far the most common reason for admission (about 30,000 visits), with a rate slightly above average (11.7%). Respiratory and musculoskeletal cases have the lowest rates.

### Extra: How do the numeric columns relate to each other?

In [38]:
corr_cols = ["time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications",
             "number_outpatient", "number_emergency", "number_inpatient",
             "number_diagnoses", "readmitted_30"]
corr = df[corr_cols].corr().round(2)

fig = px.imshow(corr, text_auto=True, color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
                title="Correlation between numeric columns")
fig.update_layout(height=600)
fig.show()

**Observation:** No single number has a strong straight-line link with readmission. The highest is `number_inpatient` (0.17), confirming Q3. `time_in_hospital` and `num_medications` are related to each other (0.46): longer stays go with more medications. Readmission depends on a **combination** of factors, which is why a machine learning model would be a good next step.

In [39]:
gender_rate = readmit_rate(df, "gender")
gender_rate

,visits,readmit_rate
gender,,
Female,53454,11.46
Male,45886,11.30


**Observation:** Women (11.5%) and men (11.3%) have almost the same rate, so gender is not a useful warning sign on its own.

## 6. Save the clean data back to MySQL
The Streamlit app reads this cleaned table (`encounters_clean`), so the dashboard always shows the same clean data used in this analysis.

In [40]:
keep_cols = ["encounter_id", "patient_nbr", "race", "gender", "age_group",
             "admission_group", "admission_source", "discharge_group", "medical_specialty",
             "time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications",
             "number_outpatient", "number_emergency", "number_inpatient", "total_prior_visits",
             "number_diagnoses", "primary_diagnosis", "a1c_result", "a1c_tested",
             "max_glu_serum", "insulin", "metformin", "med_change", "diabetes_med",
             "readmitted", "readmit_status", "readmitted_30"]

clean = df[keep_cols]
clean.to_sql("encounters_clean", con=engine, if_exists="replace",
             index=False, chunksize=5000, method="multi")
print(f"Saved {len(clean):,} rows to MySQL table 'encounters_clean'")

Saved 99,340 rows to MySQL table 'encounters_clean'


## 7. Key observations

1. **About 1 in 9 visits (11.4%) ends in a 30-day readmission.** After removing patients who died or went to hospice, 11,300 of 99,340 visits led to an early return.
2. **Previous hospital stays are the strongest warning sign.** The rate rises from 8.6% (no inpatient visits last year) to 37.1% (5 or more), more than four times higher.
3. **Discharge destination matters.** Patients sent to a care/rehab facility (16.5%) or another hospital (16.0%) return far more often than those sent home (9.3%).
4. **Longer stays carry more risk.** The rate climbs from 8.4% for 1-day stays to about 14–15% for 8–10 days.
5. **Insulin dose changes signal risk.** Increased (13.3%) or decreased (14.2%) insulin both beat the 10.2% rate of patients not on insulin.
6. **The A1C test is rarely done.** It was skipped in 83% of visits, and tested patients were readmitted less often (9.9% vs 11.7%).
7. **Older patients are the biggest group at risk.** Ages 60–90 make up about 64% of visits, with rates of 11.3–12.6%.
8. **Patients admitted for diabetes itself have the highest diagnosis-group rate** (13.1%), while circulatory problems are the most common reason for admission.
9. **Gender makes almost no difference** (11.5% vs 11.3%).

## 8. Recommendations
- **Flag frequent patients:** anyone with 2+ inpatient stays in the past year should automatically get a follow-up call or clinic visit within a week of discharge.
- **Strengthen hand-offs to care facilities:** send clear discharge notes and medication plans when patients go to rehab or nursing facilities, since these groups return most often.
- **Test A1C more often:** make the A1C test a standard part of diabetic admissions, since 83% of visits skipped it.
- **Watch insulin changes:** patients whose insulin dose changed should get extra education and an early check-up to make sure the new dose is working.
- **Next step:** build a machine learning model that combines these factors into one risk score per patient, since no single factor is strong enough alone.

## 9. Limitations
- The data is from 1999–2008, so today's hospital practices may differ.
- These are **associations, not causes**. For example, A1C testing may simply happen more often at better-resourced hospitals.
- Some patients appear many times, which gives frequent visitors more weight in the overall rates.